## Sparse+UserBatch Hybrid: Save + Predict + Metrics

This notebook:

- Loads the trained hybrid model from `hybrid_adam_pso_workers_pso_v2_sparse_userbatch_results.pt`
- Rebuilds the same collaborative filtering model architecture
- Saves a standalone checkpoint (`hybrid_sparse_userbatch_model_state_dict.pt`)
- Runs predictions on the **combined test set** from `splits.pt`
- Computes performance metrics: **MSE, RMSE, MAE, R²**


In [1]:
import os
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(1)
DEVICE = torch.device('cpu')

SPLITS_PATH = 'splits.pt'
RESULTS_PATH = 'hybrid_adam_pso_workers_pso_v2_sparse_userbatch_results.pt'
OUT_MODEL_PATH = 'hybrid_sparse_userbatch_model_state_dict.pt'

# Match the training script hyperparams
EMB_DIM = 16
HIDDEN_DIM = 16
DROPOUT = 0.1
DROPOUT_EMB = 0.2
BATCH_SIZE = 2048
MAX_SAMPLES_PER_SPLIT = 20000


In [2]:
def load_tensor_splits(max_samples_per_split: int | None = MAX_SAMPLES_PER_SPLIT):
    assert os.path.exists(SPLITS_PATH), f"{SPLITS_PATH} not found. Generate it first."
    raw_splits = torch.load(SPLITS_PATH)

    all_X_full = torch.cat(
        [torch.cat([s['X_train'], s['X_test']], dim=0) for s in raw_splits],
        dim=0,
    )
    n_users_global = int(all_X_full[:, 0].max().item()) + 1
    n_movies_global = int(all_X_full[:, 1].max().item()) + 1

    tensor_splits = []
    for s in raw_splits:
        X_train, y_train = s['X_train'], s['y_train']
        X_test, y_test = s['X_test'], s['y_test']

        if max_samples_per_split is not None and X_train.size(0) > max_samples_per_split:
            X_train = X_train[:max_samples_per_split].clone()
            y_train = y_train[:max_samples_per_split].clone()
        if max_samples_per_split is not None and X_test.size(0) > max_samples_per_split:
            X_test = X_test[:max_samples_per_split].clone()
            y_test = y_test[:max_samples_per_split].clone()

        tensor_splits.append({'X_train': X_train, 'y_train': y_train, 'X_test': X_test, 'y_test': y_test})

    return tensor_splits, n_users_global, n_movies_global


def make_combined_test_loader(tensor_splits, batch_size=BATCH_SIZE):
    X_all = torch.cat([s['X_test'] for s in tensor_splits], dim=0)
    y_all = torch.cat([s['y_test'] for s in tensor_splits], dim=0)
    ds = TensorDataset(X_all, y_all)
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


In [3]:
class SparseHybridCollabFiltering(nn.Module):
    """Same architecture as hybrid sparse+userbatch training script."""

    def __init__(self, n_users, n_movies, emb_dim=EMB_DIM, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim, sparse=True)
        self.movie_emb = nn.Embedding(n_movies, emb_dim, sparse=True)
        self.dropout_emb = DROPOUT_EMB

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
            nn.ReLU(),
        )

    def forward(self, user, movie):
        u = F.dropout(self.user_emb(user), p=self.dropout_emb, training=self.training)
        m = F.dropout(self.movie_emb(movie), p=self.dropout_emb, training=self.training)
        x = torch.cat([u, m], dim=1)
        return self.mlp(x).squeeze()


In [4]:
assert os.path.exists(RESULTS_PATH), f"Missing {RESULTS_PATH}. Generate it by running hybrid_adam_pso_workers_pso_v2_sparse_userbatch.py"

obj = torch.load(RESULTS_PATH, map_location='cpu')
state_dict = obj['state_dict']

# Load data and rebuild model with correct embedding sizes
splits, n_users_global, n_movies_global = load_tensor_splits()
print('n_users_global:', n_users_global)
print('n_movies_global:', n_movies_global)

test_loader = make_combined_test_loader(splits)

model = SparseHybridCollabFiltering(n_users_global, n_movies_global).to(DEVICE)
model.load_state_dict(state_dict)
model.eval()

# Save a standalone checkpoint
torch.save({'state_dict': state_dict, 'n_users': n_users_global, 'n_movies': n_movies_global}, OUT_MODEL_PATH)
print('Saved model checkpoint:', OUT_MODEL_PATH)


n_users_global: 1000
n_movies_global: 11459
Saved model checkpoint: hybrid_sparse_userbatch_model_state_dict.pt


C:\Users\gouth\AppData\Local\Temp\ipykernel_21764\4258469278.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(RESULTS_PATH, map_location='cpu')


In [5]:
@torch.no_grad()
def predict_all(model, loader):
    preds_all = []
    y_all = []
    for Xb, yb in loader:
        Xb = Xb.to(DEVICE)
        yb = yb.float().to(DEVICE)
        preds = model(Xb[:, 0].long(), Xb[:, 1].long()).float()
        preds_all.append(preds.cpu())
        y_all.append(yb.cpu())
    return torch.cat(preds_all), torch.cat(y_all)


def regression_metrics(preds: torch.Tensor, y: torch.Tensor):
    preds = preds.float()
    y = y.float()

    mse = torch.mean((preds - y) ** 2).item()
    rmse = math.sqrt(mse)
    mae = torch.mean(torch.abs(preds - y)).item()

    y_mean = torch.mean(y)
    ss_tot = torch.sum((y - y_mean) ** 2).item()
    ss_res = torch.sum((y - preds) ** 2).item()
    r2 = float('nan') if ss_tot == 0 else 1.0 - (ss_res / ss_tot)

    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}


preds, y = predict_all(model, test_loader)
metrics = regression_metrics(preds, y)
metrics

{'mse': 0.04948938637971878,
 'rmse': 0.22246210099636923,
 'mae': 0.17291119694709778,
 'r2': 0.12564801136327908}

In [6]:
# Show a few example predictions
k = 10
idx = torch.randperm(len(y))[:k]
examples = torch.stack([y[idx], preds[idx]], dim=1)
print('Columns: [true_rating, pred_rating]')
examples

Columns: [true_rating, pred_rating]


tensor([[0.8000, 0.7337],
        [0.8000, 0.7019],
        [0.9000, 0.7315],
        [0.6000, 0.6364],
        [0.7000, 0.7170],
        [1.0000, 0.5936],
        [1.0000, 0.5844],
        [0.8000, 0.6878],
        [0.8000, 0.6542],
        [0.9000, 0.6491]])

In [7]:
# Build combined X_test to map predictions back to users/items
X_test_all = torch.cat([s['X_test'] for s in splits], dim=0)

k = 10
idx = torch.randperm(len(y))[:k]

print("user_id, movie_id, true_rating, pred_rating")
for j in idx.tolist():
    u = int(X_test_all[j, 0].item())
    m = int(X_test_all[j, 1].item())
    print(u, m, float(y[j].item()), float(preds[j].item()))

user_id, movie_id, true_rating, pred_rating
142 1482 1.0 0.6335286498069763
1 1497 0.4000000059604645 0.5989401340484619
70 715 0.800000011920929 0.7630385160446167
69 715 1.0 0.625465452671051
348 121 1.0 0.9089027643203735
98 86 0.8999999761581421 0.7536115646362305
75 808 0.699999988079071 0.5294561386108398
624 333 0.6000000238418579 0.6069912910461426
39 931 0.20000000298023224 0.5764166116714478
592 519 0.6000000238418579 0.670925498008728


## Optional: Save predictions

If you want to analyze predictions later (e.g., calibration, per-user errors), you can save them.


In [ ]:
OUT_PRED_PATH = 'hybrid_sparse_userbatch_test_predictions.pt'

torch.save(
    {
        'y_true': y,
        'y_pred': preds,
        'metrics': metrics,
    },
    OUT_PRED_PATH,
)
print('Saved predictions:', OUT_PRED_PATH)
